In [0]:
!pip install xlrd
!pip install openpyxl

In [0]:
import pandas as pd
import glob
import re
from pyspark.sql.functions import col, datediff, when, concat, coalesce, lit, date_format,to_timestamp,udf,to_date,lower
from pyspark.sql.types import StructType, StructField, DateType, IntegerType
from datetime import timedelta, datetime

In [0]:
def extract_date_from_filename(filename):
    """Extract date from filename in format 'dd.mm.yyyy'."""
    match = re.search(r'(\d{2})\.(\d{2})\.(\d{4})', filename)
    if match:
        day, month, year = match.groups()
        return f'{year}-{month}-{day}'  # Format as YYYY-MM-DD
    return None

def add_reporting_date(file_paths):
    # Create an empty list to store DataFrames
    dfs = []
    if 'Open' in file_paths[0]:
        for file in file_paths:
            # Extract the filename
            filename = file.split('/')[-1]
            
            # Read the Excel file
            df = pd.read_excel(file, dtype=str, skiprows=7)
            
            # Extract date from filename for all files
            reporting_date = extract_date_from_filename(filename)
            df['Reporting Date'] = reporting_date
            
            dfs.append(df)
        return dfs
    else:
        for file in file_paths:
            # Extract the filename
            filename = file.split('/')[-1]
            
            # Read the Excel file
            df = pd.read_excel(file, dtype=str, skiprows=8)
            
            # Extract date from filename for all files
            reporting_date = extract_date_from_filename(filename)
            df['Reporting Date'] = reporting_date
            
            dfs.append(df)
        return dfs

def process_tickets(me_file_paths, staff_mapping_df, destination_path):
    """Process tickets and merge with existing data"""

    # Initialize an empty list to store file paths
    file_paths = []
    
    # Iterate over the list of file path patterns
    for path_pattern in me_file_paths:
        file_paths.extend(glob.glob(path_pattern))

    # Add Reporting Date Column
    dfs = add_reporting_date(file_paths)

    # Concatenate all DataFrames
    combined_df = pd.concat(dfs, ignore_index=True)

    # Drop column Unnamed: 0
    combined_df.drop('Unnamed: 0',axis=1,inplace=True)
    
    # Convert to Spark DataFrame
    input_tickets_df = spark.createDataFrame(combined_df)

    input_tickets_df = input_tickets_df.withColumn("Technician", lower(col("Technician")))

    # Drop duplicates from staff 
    staff_mapping_unique = staff_mapping_df.withColumn("Employee Full Name", lower(col("Employee Full Name")))
    staff_mapping_unique = staff_mapping_unique.dropDuplicates(["Employee Full Name"])
    
    # Join with Staff_Mapping to get Lead and Manager column
    input_tickets_df = input_tickets_df.join(
        staff_mapping_unique,
        staff_mapping_unique["Employee Full Name"] == input_tickets_df["Technician"],
        "left"
    ).drop("Employee Number", "Employee Full Name", "Status", "Scope")

    # Format the timestamp
    input_tickets_df = input_tickets_df.withColumn(
        "Created", date_format(to_timestamp(col("Created Time"), "dd/MM/yyyy hh:mm a"), "yyyy-MM-dd")
    ).withColumn(
        "Difference", datediff(col("Reporting Date"), col("Created"))
    ).withColumn(
        "Ageing",
        when((col("Difference") > -10) & (col("Difference") <= 1), "1. 0 to 1 Day")
        .when((col("Difference") > 1) & (col("Difference") <= 2), "2. 1 to 2 days")
        .otherwise("3. 2+ Days")
    ).withColumn(
        "New Ticket",
        when(col("Reporting Date") <= col("Created"), "New")
        .otherwise("Other")
    ).withColumn(
        "Last Update Gap",
        coalesce(datediff(date_format(to_timestamp(col("Last Update Time"), "dd/MM/yyyy hh:mm a"), "yyyy-MM-dd"), col("Created")), lit(0))
    ).withColumnRenamed("Team Lead", "Lead").withColumnRenamed("Line Manager", "Manager").withColumn(
        "Lead", coalesce(col("Lead"), lit('Unassigned'))
    ).withColumn(
        "Manager", coalesce(col("Manager"), lit('Unassigned'))
    )
    # Write the data to the destination
    input_tickets_df.write.mode("overwrite").parquet(destination_path)
    print(f"Data has been successfully processed and written at {destination_path}")

In [0]:
# main path
source_path = "/dbfs/mnt/stppeedp/ppeedp/landing/data0/staging/eag/ey/ap_automation/"

# Define the path to write the output 
destination_path_open = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/open_tickets'
destination_path_resolved = 'dbfs:/mnt/stppeedp/ppeedp/prod/eag/ey/fdw/mfr/resolved_tickets'

staff_mapping_path = source_path+'input_files/Staff Mapping.xlsx'

# List all files in manage_engine directory
all_files = [f.path for f in dbutils.fs.ls(source_path.replace('/dbfs','') + "manage_engine/")]

# Define regex patterns for filtering
resolved_pattern = re.compile(r"resolved", re.IGNORECASE)
open_pattern = re.compile(r"open", re.IGNORECASE)

# Filter case-insensitively
me_resolved_paths = [f.replace('dbfs:','/dbfs') for f in all_files if resolved_pattern.search(f)]
me_open_paths = [f.replace('dbfs:','/dbfs') for f in all_files if open_pattern.search(f)]

In [0]:
# Reading Dynamic Tables - Staff Mapping, 
staff_mapping_df = pd.read_excel(staff_mapping_path, dtype=str)
staff_mapping = spark.createDataFrame(staff_mapping_df)

In [0]:
# Process Open Tickets
process_tickets(me_open_paths, staff_mapping, destination_path_open)

# Process Resolved Tickets
process_tickets(me_resolved_paths, staff_mapping, destination_path_resolved)